# Sesión 1 — Qué es Machine Learning y el pipeline completo en 30 líneas

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 1

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Explicar qué es *aprender de datos* y distinguir aprendizaje supervisado de no supervisado con un ejemplo propio.
2. Identificar en un problema real cuál es la variable objetivo (*y*) y cuáles son las variables de entrada (*X*).
3. Ejecutar de punta a punta un pipeline de ML en Python: datos → preprocesamiento → modelo → evaluación → interpretación.
4. Comparar un modelo contra una *línea base trivial* y decir si el modelo aprendió algo.

> 💾 **Antes de escribir nada:** menú *Archivo → Guardar una copia en Drive* y trabaje en esa copia; si no, lo que escriba se pierde al cerrar la pestaña. Si Colab avisa que el notebook no es de Google, elija *Ejecutar de todos modos*.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
   HOY: recorremos las cinco etapas de pasada. En las próximas sesiones profundizamos una por una.
```

## Cómo usar este notebook
- 🔍 **Prediga**: antes de ejecutar la celda, escriba su predicción. No es examen, es para que su cerebro se comprometa con una respuesta antes de ver la real.
- ✏️ **Complete**: hay líneas con `...` y `# TODO`. Reemplácelas.
- 💥 **Trampa**: algo que sale "bien" pero está mal. Hoy no hay; a partir de la Sesión 3 sí.
- 🖼️ **Figuras**: cada algoritmo se *ve* antes de correrse (cómo funciona por dentro), después de entrenarlo (qué aprendió con nuestros datos) y al cambiarle su parámetro clave. Debajo de cada figura hay una frase que dice qué mirar.
- ⚡ **Mini-reto**: opcional, para quien vaya rápido.

> **Datos de hoy:** incidentes viales reportados en Medellín entre 2019 y agosto de 2025 (Secretaría de Movilidad de Medellín; conjunto de datos publicado por Sánchez, Arango y Correa en la revista *Data*, 2026, licencia CC BY 4.0). Cada fila es un incidente: fecha, hora, clase (choque, atropello…), gravedad (solo daños, herido, muerto), comuna, barrio y coordenadas.


## 0. Setup

Ejecute estas dos celdas tal cual. Cargan las librerías y consiguen los datos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)

# Los datos y las figuras del curso viven en el repositorio del curso.
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVO = "incidentes_medellin_2019_2025.csv"
URL_REPO = REPO + "datos/" + ARCHIVO


<details><summary>Celda de plomería (no hay que entenderla hoy): consigue los datos y las figuras del curso</summary>

Baja al disco de Colab (una sola vez por sesión) el CSV y `figuras.py`, un archivo del curso con las funciones que dibujan los algoritmos. Si el repositorio no responde, use el Plan B de abajo.
</details>

In [ ]:
def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo} del repositorio. Use el Plan B (celda siguiente).")
    return False

asegurar_datos(ARCHIVO, URL_REPO)
asegurar_datos("figuras.py", REPO + "tools/figuras.py")
URL = ARCHIVO
try:
    import figuras as fg   # las figuras del curso: fg.pipeline(), fg.arbol_paso_a_paso(), ...
except ModuleNotFoundError:   # Plan B: GitHub no respondió y no llegó figuras.py
    from google.colab import files
    print("⚠️ No llegó figuras.py: súbalo (el profesor lo comparte por el canal del curso).")
    files.upload()
    import figuras as fg
print("Listo. pandas", pd.__version__, "| datos en:", URL)

In [ ]:
fg.pipeline("todas", modulo=1, subtitulo="HOY: recorremos las cinco etapas de pasada; en las próximas sesiones profundizamos una por una");

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar el archivo</summary>

La celda siguiente revisa sola si los datos llegaron; si no, le pedirá subir `incidentes_medellin_2019_2025.csv` desde su computador (el instructor lo comparte por el canal del curso) y a partir de ahí todo el notebook funciona igual.
</details>

In [ ]:
# Plan B: solo actúa si la descarga falló. Con "Ejecutar todo" no se detiene.
from pathlib import Path
if Path(URL).exists():
    print("✓ Datos disponibles: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ No se encontró", URL, "→ suba el archivo que compartió el instructor.")
    subido = files.upload()
    if subido:
        URL = next(iter(subido))
        print("Usando archivo local:", URL)

## 1. Datos — cargar y mirar

### 👀 Mire
El archivo tiene **un incidente por fila** y cubre casi 7 años de una ciudad de ~2,6 millones de habitantes. Antes de correr la celda, piense un segundo: ¿cuántas filas tendrá (miles, decenas de miles, cientos de miles)? ¿Y cuántas columnas (mire la descripción de los datos arriba)?

In [ ]:
# read_csv si es CSV; read_excel si alguien subió el Excel original (tarda 2-3 min)
df = pd.read_excel(URL) if URL.endswith(".xlsx") else pd.read_csv(URL)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

### 1.1 Primer paso de limpieza: los nombres de las columnas

Mire los nombres: `LLAVE`, `AÑO`, `Latitud`, `Codigo Comuna Planeacion`… Unos en mayúscula, otros en minúscula, con espacios y con tildes. Así vienen casi todos los datos reales, y es una fuente inagotable de errores de tipeo (`df["Comuna Planeacion"]` vs `df["comuna_planeacion"]`).

La convención que usaremos todo el curso es **`snake_case`**: minúsculas, sin tildes ni `ñ`, y `_` en vez de espacios. Es la que usan pandas, scikit-learn y la mayoría de bases de datos.

### 👀 Mire
Después de aplicar la regla, ¿cómo quedará `Codigo Comuna Planeacion`? ¿Y `AÑO`? Piénselo y compruébelo en la salida.

In [ ]:
import unicodedata

def limpiar_nombre(nombre):
    """'Codigo Comuna Planeacion' -> 'codigo_comuna_planeacion'; 'AÑO' -> 'ano'."""
    nombre = nombre.strip().lower()                                   # espacios al borde, minúsculas
    nombre = unicodedata.normalize("NFKD", nombre)                    # separa tildes y ñ
    nombre = nombre.encode("ascii", "ignore").decode()                # ...y las elimina
    return nombre.replace(" ", "_")                                   # espacios -> _

df.columns = [limpiar_nombre(c) for c in df.columns]
list(df.columns)

Ya son consistentes, pero siguen siendo largos. Un segundo paso, opcional pero cómodo, es acortarlos con `rename` a nombres que se escriban sin pensar. Fíjese que `ano` (año sin ñ) queda feo: lo renombramos a `anio`, que es la convención habitual en datos colombianos.

In [ ]:
df = df.rename(columns={
    "llave": "id", "ano": "anio", "fecha_incidente": "fecha", "hora_incidente": "hora",
    "clase_incidente": "clase", "gravedad_incidente": "gravedad", "direccion_interpolada": "direccion",
    "codigo_comuna_planeacion": "codigo_comuna", "comuna_planeacion": "comuna",
    "codigo_barrio_planeacion": "codigo_barrio", "barrio_planeacion": "barrio",
})
df = df[df["anio"] >= 2019]        # el curso trabaja con 2019 en adelante (no cambia nada si ya venía recortado)
print(df.shape)
df.head(3)

## Pausa conceptual — ¿qué es aprender de datos?

**Machine Learning** es construir un programa que, en vez de recibir las reglas escritas por nosotros, *encuentra* las reglas a partir de ejemplos.

| Tipo | Tenemos… | Queremos… | Con estos datos |
|---|---|---|---|
| **Supervisado — clasificación** | ejemplos con su etiqueta (categoría) | predecir la etiqueta de un caso nuevo | ¿este incidente tendrá víctimas? |
| **Supervisado — regresión** | ejemplos con su valor numérico | predecir el número de un caso nuevo | ¿cuántos incidentes habrá en El Poblado el viernes a las 6 p.m.? |
| **No supervisado** | ejemplos *sin* etiqueta | descubrir estructura: grupos, patrones | ¿qué comunas se parecen en su perfil de incidentes? |

Dos palabras que usaremos todo el curso:
- **X** — las variables de entrada (*features*): lo que sabemos *antes* de que pase lo que queremos predecir.
- **y** — la variable objetivo (*target*): lo que queremos predecir.

Por dentro, aprender es siempre lo mismo: una **familia de modelos** (hoy, árboles), una **medida de error** que se quiere bajar (hoy, la impureza de Gini) y una **búsqueda** que prueba opciones hasta encontrar la que más la baja.

Hoy vamos a hacer **clasificación**: predecir si un incidente tuvo víctimas (heridos o muertos) o solo daños materiales, a partir de *cuándo*, *dónde* y *qué tipo* de incidente fue.

### ✏️ Completa — ¿qué columna sería la *y*?

Escriba aquí, en una frase, **qué le gustaría predecir** con estos datos y **qué columna** lo contiene.
Hay más de una respuesta válida.

**Su respuesta:** ____

<details><summary>Pistas</summary>

- Si quiero predecir *cuántos* incidentes habrá en una comuna a cierta hora → eso es una cantidad → **regresión** (pero ojo: esa columna no existe todavía, habría que construirla).
- Si quiero predecir *qué tan grave* será un incidente → eso es una categoría → **clasificación**.
- Si quiero *agrupar* comunas parecidas sin una respuesta correcta → **no supervisado**.
</details>

## 2. Preprocesamiento — dejar los datos en forma de X e y

Un modelo de scikit-learn necesita una **tabla numérica sin nulos** (X) y un **vector objetivo** (y). Los datos crudos casi nunca vienen así. Hoy hacemos lo mínimo; la Sesión 2 es entera sobre esto.

### 2.1 Construir la variable objetivo

In [ ]:
# y = 1 si el incidente tuvo víctimas (heridos o muertos), 0 si fue solo daños
df["con_victimas"] = (df["gravedad"] != "SOLO DAÑOS").astype(int)
df["gravedad"].value_counts()

### 🔍 Predice
Con la tabla de arriba: ¿qué **porcentaje** de incidentes tuvo víctimas?

**Su predicción:** ____ %

In [ ]:
df["con_victimas"].mean().round(3)

### 2.2 Construir variables de entrada a partir de la fecha y la hora

`fecha` y `hora` son texto (`"2019-01-01"`, `"00:10"`). Un árbol no sabe leer texto: hay que convertirlas en números que tengan sentido: hora del día (los dos primeros caracteres de `hora`), día de la semana, mes.

### ✏️ Completa
Complete las dos líneas marcadas con `TODO`. Pista: los accesores `.dt.dayofweek` y `.dt.month` sobre `df["fecha"]`.

In [ ]:
df["fecha"] = pd.to_datetime(df["fecha"])
df["hora_num"] = df["hora"].astype(str).str.slice(0, 2).astype(int)   # "07:35" -> 7
df["dia_semana"] = ...   # TODO  (0 = lunes … 6 = domingo)
df["mes"] = ...          # TODO

df[["fecha", "hora", "hora_num", "dia_semana", "mes"]].head()

### 2.3 Nulos y texto → números

Mire otra vez el `df.info()`: `latitud` y `barrio` tienen vacíos, y en `comuna` hay un valor `N/D` que es un nulo disfrazado de texto. Hoy esquivamos el problema usando solo columnas completas; en la Sesión 2 lo enfrentamos. Dos operaciones estándar:
- **Quitar filas con nulos** en las columnas que vamos a usar (`dropna`). Es la solución más burda; la Sesión 2 muestra alternativas. Con las seis columnas que elegimos hoy no hay vacíos, así que va a descartar **0 filas**: la dejamos porque es el hábito (si mañana agrega `latitud`, que sí tiene vacíos, `dropna` empieza a quitar filas).
- **One-hot encoding** (`pd.get_dummies`): convertir una columna de texto con *k* categorías en *k* columnas de 0/1.

In [ ]:
columnas_x = ["hora_num", "dia_semana", "mes", "anio", "comuna", "clase"]
datos = df[columnas_x + ["con_victimas"]].dropna()

X = pd.get_dummies(datos[columnas_x], columns=["comuna", "clase"], dtype=int)
y = datos["con_victimas"]

print("Filas descartadas por nulos:", len(df) - len(datos))
print("X:", X.shape, "  y:", y.shape)
X.head(3)

### 👀 Mire
`X` pasó de 6 columnas a las que ve arriba. ¿De dónde salieron? La celda siguiente cuenta las comunas distintas (incluyendo `N/D`) y las clases: compruebe que 4 numéricas + comunas + clases da el total.

In [ ]:
print("comunas:", datos["comuna"].nunique(), " clases:", datos["clase"].nunique())

## 3. Modelo — separar, entrenar

### La regla que no se negocia: separar *antes* de entrenar

Vamos a partir los datos en dos: **entrenamiento** (el modelo aprende de aquí) y **prueba** (aquí medimos qué tan bien predice casos que *nunca vio*).

### 👀 Piense (no hay celda que ejecutar)
¿Por qué no entrenamos con **todos** los datos y luego medimos con esos mismos? Escriba su hipótesis en una frase.

**Su respuesta:** ____

*(Guarde la respuesta: hoy mismo, en la sección 4.1, la comprobamos con números.)*

> 🧪 **Laboratorio** (para jugar después de clase, o ya si va adelantado): [**Árbol de decisión**](https://nitobest.github.io/curso-aad-python/laboratorio/arbol.html) · cómo crece pregunta a pregunta, usted elige el corte y compara su Gini con el del árbol, y el reto de la profundidad. Tres pestañas: **Mira**, **Juega** y **Reto** (prediga antes de correr). Funciona en el celular.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Entrenamiento:", X_train.shape[0], "filas   Prueba:", X_test.shape[0], "filas")

### El modelo de hoy: un árbol de decisión

Un **árbol de decisión** aprende una secuencia de preguntas de sí/no sobre X (*¿hora_num ≤ 5.5?*, *¿clase_Atropello = 1?*…) que separe lo mejor posible las clases de y. Lo elegimos hoy porque **se puede leer**: al final vamos a ver las preguntas que aprendió.

#### Antes del código: cómo piensa un árbol

Vamos a **verlo** con datos inventados de dos variables (`hora` y `velocidad`) y dos clases (rojo = con víctimas, azul = solo daños). Cada pregunta del árbol corta el plano con una línea recta.

### 🔍 Predice
Un árbol de **profundidad 3** hace hasta 3 preguntas seguidas antes de decidir. ¿En cuántas regiones (rectángulos) **como máximo** puede partir el plano?

**Su predicción:** ____ regiones

In [ ]:
fg.arbol_paso_a_paso();

Las regiones del mapa y las preguntas del árbol son **lo mismo dibujado de dos maneras**. A la izquierda, el árbol como diagrama de flujo; a la derecha, como mapa. Cada hoja tiene un número y ese número marca su rectángulo. Un incidente nuevo (la ★: 3 a. m., 80 km/h) responde las preguntas de arriba abajo y cae en una hoja: esa es la predicción.

In [ ]:
fg.arbol_recorrido(fila=(3, 80));

#### ¿Qué tan revuelta está una caja? La impureza de Gini con pelotas

Imagine cada región como una caja con pelotas: **rojas** = con víctimas, **azules** = solo daños. El árbol quiere preguntas que dejen cajas de **un solo color**. Para medir qué tan revuelta está una caja se usa la **impureza de Gini**: $G = 1 - p_{\text{rojo}}^2 - p_{\text{azul}}^2$. Mire la cuenta hecha a mano en tres cajas:

In [ ]:
fg.gini_pelotas();

Ahora la misma caja de 10 incidentes partida por **dos preguntas distintas**. Cada lado tiene su propio Gini y se promedian **pesando por el tamaño** de cada lado (Gini ponderado). Gana la pregunta con el número más bajo.

In [ ]:
fg.gini_dos_cortes();

¿Y de dónde salen las preguntas candidatas? De los datos: entre cada par de valores vecinos hay un corte posible. Con 10 incidentes hay 9 cortes en `hora` (y otros 9 en `velocidad`). El árbol calcula el Gini ponderado de **todos** y se queda con el más bajo:

In [ ]:
fg.arbol_mini_datos();

Con 300 incidentes es exactamente lo mismo, solo que con cientos de cortes candidatos: la curva es el Gini ponderado de cada corte posible y el árbol elige el fondo del valle.

In [ ]:
fg.arbol_como_elige_el_corte();

### 📐 Fundamento: la fórmula detrás de la figura

Lo que la curva de arriba mide en cada punto es la **impureza de Gini** de una región (una hoja del árbol). Si en la región hay una proporción $p_1$ de incidentes *con víctimas* y $p_0 = 1 - p_1$ *sin víctimas*:

$$G = 1 - \sum_{k} p_k^2 \;=\; 1 - p_1^2 - p_0^2$$

Léala así: **$G$ es la probabilidad de equivocarse si etiqueta un incidente al azar según las proporciones de la región.**

- Región pura (solo daños, $p_1 = 0$): $G = 1 - 0 - 1 = 0$. No hay nada que aprender ahí.
- Región mitad y mitad ($p_1 = 0{,}5$): $G = 1 - 0{,}25 - 0{,}25 = 0{,}5$, el máximo posible con dos clases.
- Nuestros datos completos ($p_1 \approx 0{,}71$): $G = 1 - 0{,}71^2 - 0{,}29^2 \approx 0{,}41$. Ese es el punto de partida; cada corte intenta bajarlo.

<details><summary><b>Para quien quiera más:</b> cómo se compara un corte con otro</summary>

Un corte parte la región en izquierda (con $n_I$ incidentes) y derecha ($n_D$). El árbol no mira una sola mitad sino el **promedio ponderado** de las dos impurezas:

$$G_{\text{corte}} = \frac{n_I}{n}\,G_I + \frac{n_D}{n}\,G_D$$

y elige el umbral, entre todas las variables y todos los valores posibles, que hace $G_{\text{corte}}$ mínimo (equivalente: que maximiza la *ganancia* $G - G_{\text{corte}}$). Eso es lo que dibuja la curva: un $G_{\text{corte}}$ por cada umbral candidato; el fondo del valle es el ganador.

¿Por qué Gini y no simplemente "cuántos acierto"? Porque el accuracy de un corte a menudo no cambia aunque el corte mejore: si los dos lados conservan la misma clase mayoritaria, se aciertan los mismos casos antes y después. Gini, en cambio, sí premia que un lado quede más puro aunque no cambie la clase mayoritaria. scikit-learn usa Gini por defecto (`criterion="gini"`); la alternativa clásica es la **entropía**, $H = -\sum_k p_k \log_2 p_k$, que ordena los cortes casi igual.
</details>

Verifíquelo con una línea: el Gini de todo `y_train` debería salir cerca de 0,41.

In [ ]:
p1 = y_train.mean()            # proporción con víctimas
gini = 1 - p1**2 - (1 - p1)**2
print(f"p1 = {p1:.3f}   Gini(y_train) = {gini:.3f}")

### ✏️ Completa
Toda la API de scikit-learn tiene la misma forma: se crea el modelo, se le llama `.fit(X_train, y_train)`, y luego `.predict(X_nuevo)`. Complete el `fit`.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo.fit(...)   # TODO: ¿con qué X y qué y se entrena?
print("Modelo entrenado.")

## 4. Evaluación — ¿aprendió algo?

**Accuracy** (exactitud) = proporción de predicciones correctas. Es la métrica más simple y hoy nos sirve.

$$\text{accuracy} = \frac{\#\,\text{aciertos}}{\#\,\text{incidentes en prueba}} = \frac{1}{n}\sum_{i=1}^{n} \mathbb{1}[\hat{y}_i = y_i]$$

donde $\hat{y}_i$ es lo que predijo el modelo para el incidente $i$, $y_i$ lo que pasó de verdad, y $\mathbb{1}[\cdot]$ vale 1 si coinciden y 0 si no. La que reportamos se calcula sobre los datos de **prueba**, que el modelo no vio; sobre entrenamiento sirve para diagnosticar (lo verá en la sección 4.1).

### 🔍 Predice
Antes de correr la celda:
1. ¿Qué accuracy tendrá un "modelo" tonto que **siempre** predice la clase mayoritaria? Pista: usted calculó ese número en la sección 2.1.
2. ¿Cree que nuestro árbol le gana? ¿Por cuánto?

**Su predicción:** tonto ≈ ____ · árbol ≈ ____

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# Línea base: siempre la clase más frecuente
tonto = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

acc_tonto = accuracy_score(y_test, tonto.predict(X_test))
acc_arbol = accuracy_score(y_test, modelo.predict(X_test))

print(f"Accuracy línea base (siempre la clase mayoritaria): {acc_tonto:.3f}")
print(f"Accuracy árbol de decisión:                         {acc_arbol:.3f}")

### Hábito del curso
Todo modelo, en todas las sesiones, se compara contra una línea base trivial (`DummyClassifier` / `DummyRegressor`). Si no le gana, no hay modelo.

### 4.1 ¿Y si el árbol fuera más profundo?

Usamos `max_depth=4` sin justificarlo. La figura de abajo entrena el árbol con profundidad 1, 2, 4, 8, 16 y sin límite, y mide el accuracy dos veces: sobre los datos de **entrenamiento** (los que el árbol ya vio) y sobre los de **prueba** (nunca vistos).

### 🔍 Predice
Dibuje mentalmente las dos curvas (accuracy vs profundidad):
1. La de entrenamiento, ¿sube, baja o se aplana al aumentar la profundidad?
2. La de prueba, ¿hace lo mismo?

**Su predicción:** entrenamiento ____ · prueba ____

In [ ]:
fg.arbol_profundidades(X_train, y_train, X_test, y_test);

<details><summary>¿Y la pregunta de la sección 3 (por qué no medir con los mismos datos)?</summary>

Mire la figura: si eligiéramos la profundidad mirando la curva azul (entrenamiento), escogeríamos el árbol **sin límite**, que es justamente el que peor le va con datos nuevos en la curva roja. Medir con lo que el modelo ya vio premia memorizar. Por eso el examen (prueba) se guarda aparte desde el principio.
</details>

## 5. Interpretación — ¿qué aprendió?

Un número de accuracy no cuenta la historia. Miremos **qué preguntas** hace el árbol y **qué variables** le importan.

Primero el árbol que acabamos de entrenar, dibujado (solo los dos primeros niveles, para que se lea):

In [ ]:
fg.arbol_reglas(modelo, X.columns, max_depth=2);

### Una fila entra, una predicción sale
El árbol entrenado es una **función**: le damos un incidente de prueba (un choque que nunca vio) y vemos qué preguntas contesta hasta llegar a su hoja.

In [ ]:
fila = X_test[X_test["clase_Choque"] == 1].iloc[0]
fg.camino_de_una_fila(modelo, fila, X.columns);

### ✏️ Completa — lea el árbol
1. ¿Cuál es la **primera** pregunta que hace? ¿Le sorprende?
2. Busque la pregunta sobre `anio`. ¿Qué cree que pasó en 2023 para que el árbol la considere tan útil?

**Su respuesta:** ____

Ahora la misma idea de los rectángulos, pero sobre los datos reales. A la izquierda, la proporción real de incidentes con víctimas por hora y día de la semana; a la derecha, lo que predice un árbol pequeño que **solo** ve esas dos variables, con el borde de cada una de sus hojas.

In [ ]:
fg.mapa_hora_dia(datos, profundidad=3);

En texto, las mismas reglas del árbol completo (útil para copiar y pegar en un informe):

In [ ]:
from sklearn.tree import export_text

print(export_text(modelo, feature_names=list(X.columns), max_depth=2, show_weights=False))

In [ ]:
importancias = (pd.Series(modelo.feature_importances_, index=X.columns)
                  .sort_values(ascending=False)
                  .head(10))

fig, ax = plt.subplots(figsize=(8, 4))
importancias.sort_values().plot.barh(ax=ax)
ax.set_title("¿Qué variables usa el árbol para decidir?")
ax.set_xlabel("importancia (suma a 1 sobre todas las variables)")
plt.tight_layout()

*Qué mirar: cuánto de la "mejora de Gini" del árbol se debe a cada variable. Dos barras se llevan casi todo; fíjese cuáles son y si alguna le sorprende.*

### ✏️ Completa — interpretación en dos frases

1. ¿Cuál es la variable más importante? ¿Tiene sentido con lo que usted sabe de la ciudad y del tránsito?
2. Si el árbol dice que `clase_Choque` (o `clase_Atropello`) es lo que más importa para predecir *víctimas*, ¿eso es un hallazgo o una obviedad? ¿Y si dice que la **hora** importa? ¿Cuál de los dos "descubrimientos" le contaría a la Secretaría de Movilidad?

**Su respuesta:** ____

## 6. El pipeline completo en 30 líneas

Lo que hicimos por partes, ahora seguido. **Esto es lo que va a repetir el resto del curso**; lo que cambia sesión a sesión es la profundidad de cada bloque.

In [ ]:
# ---------- 1. DATOS
df = pd.read_excel(URL) if URL.endswith(".xlsx") else pd.read_csv(URL)

# ---------- 2. PREPROCESAMIENTO
df.columns = [limpiar_nombre(c) for c in df.columns]
df = df.rename(columns={"llave": "id", "ano": "anio", "fecha_incidente": "fecha", "hora_incidente": "hora",
                        "clase_incidente": "clase", "gravedad_incidente": "gravedad",
                        "comuna_planeacion": "comuna", "barrio_planeacion": "barrio"})
df = df[df["anio"] >= 2019]
df["fecha"] = pd.to_datetime(df["fecha"])
df["hora_num"] = df["hora"].astype(str).str.slice(0, 2).astype(int)   # "07:35" -> 7
df["dia_semana"] = df["fecha"].dt.dayofweek
df["mes"] = df["fecha"].dt.month
df["con_victimas"] = (df["gravedad"] != "SOLO DAÑOS").astype(int)

columnas_x = ["hora_num", "dia_semana", "mes", "anio", "comuna", "clase"]
datos = df[columnas_x + ["con_victimas"]].dropna()
X = pd.get_dummies(datos[columnas_x], columns=["comuna", "clase"], dtype=int)
y = datos["con_victimas"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ---------- 3. MODELO
modelo = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
tonto = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

# ---------- 4. EVALUACIÓN
print(f"línea base: {accuracy_score(y_test, tonto.predict(X_test)):.3f}")
print(f"árbol:      {accuracy_score(y_test, modelo.predict(X_test)):.3f}")

# ---------- 5. INTERPRETACIÓN
print(pd.Series(modelo.feature_importances_, index=X.columns).sort_values(ascending=False).head(5))

## ⚡ Mini-reto (opcional)

Cambie la variable objetivo: en vez de *con víctimas* vs *solo daños*, prediga **con muertos** vs *todo lo demás*.

```python
df["con_muertos"] = (df["gravedad"] == "MUERTO").astype(int)
```

Repita el pipeline de la sección 6 con esa `y`. Compare el accuracy del árbol con el de la línea base y responda: **¿el árbol aprendió algo?** ¿Está seguro?

<details><summary>Pista</summary>Mire primero <code>df["con_muertos"].mean()</code>.</details>

In [ ]:
# Espacio para el mini-reto


## Cierre

Lo que hicimos hoy, en una línea por etapa:

| Etapa | Hoy | Se profundiza en |
|---|---|---|
| Datos | `read_csv` desde una URL, mirar shape/head/info | S2 |
| Preprocesamiento | nombres de columnas a `snake_case`, fecha/hora → números, `dropna`, `get_dummies` | S2 |
| Modelo | `train_test_split`, `DecisionTreeClassifier.fit` | S3–S6 |
| Evaluación | accuracy vs `DummyClassifier` | S3, S5–S6, S10 |
| Interpretación | reglas del árbol, importancia de variables | S6, S9 |

**Para la próxima sesión (S2 — explorar y preprocesar):**
- Vuelva a correr este notebook solo. Si algo no corre, anótelo y tráigalo.
- Empiece a pensar en un dataset para su proyecto: la lista curada está en el Notebook 0. La primera etapa (E1) se trabaja en la Sesión 3 y se entrega antes de la Sesión 4, por el canal que indique el profesor.
